<div style="width: 50%;">
    <img src="../ONS_Logo_Digital_Colour_Landscape_English_RGB.svg" alt="ONS Logo">
</div>

# ClassifAI Demo

This demo uses a mock occupations dataset to show how ClassifAI matches unlabelled job descriptions to SOC codes using an existing knowledgebase.

In [ ]:
import re
import pandas as pd

from classifai.indexers import VectorStore
from classifai.indexers.dataclasses import VectorStoreSearchInput
from classifai.indexers.hooks import (
    CapitalisationStandardisingHook,
    DeduplicationHook,
)
from classifai.indexers.hooks.hook_factory import HookBase

from demo_utils import (
    KNOWLEDGEBASE_PATH,
    load_uncoded_input,
    load_vectoriser,
    make_query_input,
    prepare_knowledgebase,
)

prepare_knowledgebase()
uncoded_input = load_uncoded_input()
vectoriser = load_vectoriser()


basic_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
)

---
### Example Responses

The examples below use a small batch of uncoded occupation descriptions. In a real workflow, these could come from survey responses, form submissions, or operational data waiting to be coded.

In this demo, we have the following `uncoded_input` which is a batch of free text responses we want to match to SOC codes.

In [ ]:
uncoded_input[["id", "query"]]

,id,query
0,0,Tomato Farmer: Cultivates and harvests tomatoe...
1,1,Cow Farmer: Manages dairy and beef cattle oper...
2,2,Machine Learning Engineer: Designs and deploys...
3,3,Construction Worker: Works on building sites u...
4,4,"Electrician: Installs and repairs wiring, ligh..."


---
### 1. Return A Best Match

At the simplest level, you pass in uncoded text: `query_text` 

and get back the highest-ranked suggestion for each response: `doc_label`, `doc_text`.

In [ ]:
basic_vectorstore.search(query=uncoded_input, n_results=1)

,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Vegetable farmer: Cultivates and harvests vege...,1,0.844008
1,1,Cow Farmer: Manages dairy and beef cattle oper...,5111,Dairy farmer: Manages cows for milk production...,1,0.894646
2,2,Machine Learning Engineer: Designs and deploys...,2134,"Software developer: Designs, writes, and tests...",1,0.737518
3,3,Construction Worker: Works on building sites u...,9129,Construction laborer: Performs physical tasks ...,1,0.852508
4,4,"Electrician: Installs and repairs wiring, ligh...",5241,"Electrician: Installs, maintains, and repairs ...",1,0.967119


---
### 2. Return A Shortlist For Review

When a single suggestion is not enough, the same search can return a ranked shortlist for review.

In [ ]:
basic_vectorstore.search(query=uncoded_input, n_results=2)

,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Vegetable farmer: Cultivates and harvests vege...,1,0.844008
1,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Fruit farmer: Grows and harvests fruits such a...,2,0.820159
2,1,Cow Farmer: Manages dairy and beef cattle oper...,5111,Dairy farmer: Manages cows for milk production...,1,0.894646
3,1,Cow Farmer: Manages dairy and beef cattle oper...,5111,Vegetable farmer: Cultivates and harvests vege...,2,0.738995
4,2,Machine Learning Engineer: Designs and deploys...,2134,"Software developer: Designs, writes, and tests...",1,0.737518
5,2,Machine Learning Engineer: Designs and deploys...,2134,Web developer: Builds and maintains websites a...,2,0.681587
6,3,Construction Worker: Works on building sites u...,9129,Construction laborer: Performs physical tasks ...,1,0.852508
7,3,Construction Worker: Works on building sites u...,5313,"Carpenter: Constructs, installs, and repairs w...",2,0.787516
8,4,"Electrician: Installs and repairs wiring, ligh...",5241,"Electrician: Installs, maintains, and repairs ...",1,0.967119
9,4,"Electrician: Installs and repairs wiring, ligh...",5315,"Plumber: Installs and repairs water, gas, and ...",2,0.780550


---
### 3. Return Extra Context With Each Match

The matched knowledgebase `doc_label`'s and `doc_text` is useful on its own, but sometimes extra linked information is desireable.

Extra columns such as `sector` in this example, can support analyst review, help group similar results, or allow downstream postprocessing.

In [ ]:
metadata_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    meta_data={"sector": str},  # we specify what extra context we want here.
)

metadata_vectorstore.search(query=uncoded_input, n_results=1)

,query_id,query_text,doc_label,doc_text,rank,score,sector
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Vegetable farmer: Cultivates and harvests vege...,1,0.844008,Agriculture
1,1,Cow Farmer: Manages dairy and beef cattle oper...,5111,Dairy farmer: Manages cows for milk production...,1,0.894646,Agriculture
2,2,Machine Learning Engineer: Designs and deploys...,2134,"Software developer: Designs, writes, and tests...",1,0.737518,Technology
3,3,Construction Worker: Works on building sites u...,9129,Construction laborer: Performs physical tasks ...,1,0.852508,Construction
4,4,"Electrician: Installs and repairs wiring, ligh...",5241,"Electrician: Installs, maintains, and repairs ...",1,0.967119,Construction


---
### 4. Use `search_preprocess` Hooks To Handle Messy Input

Hooks let you adapt behaviour without rewriting the search call.

Here the input is deliberately inconsistent. `CapitalisationStandardisingHook` normalises the text before embedding. 
>This is useful as data from different sources may having varying cases, leading to slight differences in matching.

In [ ]:
messy_input = make_query_input([
    "TOMATO FARMER",
    "Machine Learning ENGINEER",
    "pHd StUdEnT",
])

hook_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    hooks={
        "search_preprocess": CapitalisationStandardisingHook(method="lower")
    },  # we add any hooks here.
)


basic_results = basic_vectorstore.search(query=messy_input, n_results=1)

capitalisation_hook_results = hook_vectorstore.search(query=messy_input, n_results=1)

print("Basic results")
display(basic_results)

print("After Capitalisation Hook")
display(capitalisation_hook_results)

Basic results


,query_id,query_text,doc_label,doc_text,rank,score
0,0,TOMATO FARMER,5111,Vegetable farmer: Cultivates and harvests vege...,1,0.769499
1,1,Machine Learning ENGINEER,2134,"Software developer: Designs, writes, and tests...",1,0.702236
2,2,pHd StUdEnT,2119,University Research Assistant: Conducts univer...,1,0.642867


After Capitalisation Hook


,query_id,query_text,doc_label,doc_text,rank,score
0,0,tomato farmer,5111,Vegetable farmer: Cultivates and harvests vege...,1,0.769499
1,1,machine learning engineer,2134,"Software developer: Designs, writes, and tests...",1,0.702236
2,2,phd student,2119,University Research Assistant: Conducts univer...,1,0.642867


---
### 5. Use `search_postprocess` hooks to clean up the shortlist.

If several knowledgebase entries share the same label, the raw shortlist can contain repeated labels. 

`DeduplicationHook` trims that down to one best match per label
> This is useful as it reduces the number of matches a coder will have to look through to assign a code.

In [ ]:
dedup_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    hooks={
        "search_postprocess": DeduplicationHook(score_aggregation_method="max")
    },  # we add any hooks here.
)

review_query = make_query_input([
    "Tomato Farmer: Cultivates and harvests tomatoes in large greenhouse operations."
])

basic_results = basic_vectorstore.search(query=review_query, n_results=5)
clean_results = dedup_vectorstore.search(query=review_query, n_results=5)

print("Basic results")
display(basic_results)

print("After deduplication")
display(clean_results)

Basic results


,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Vegetable farmer: Cultivates and harvests vege...,1,0.816154
1,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Fruit farmer: Grows and harvests fruits such a...,2,0.801251
2,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Dairy farmer: Manages cows for milk production...,3,0.791010
3,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,"Sheep farmer: Raises sheep for wool, meat, and...",4,0.755043
4,0,Tomato Farmer: Cultivates and harvests tomatoe...,5313,"Carpenter: Constructs, installs, and repairs w...",5,0.661157


After deduplication


,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Vegetable farmer: Cultivates and harvests vege...,1,0.816154
1,0,Tomato Farmer: Cultivates and harvests tomatoe...,5313,"Carpenter: Constructs, installs, and repairs w...",2,0.661157


---
### 6. Add a Custom Domain-Specific Hook

Built-in hooks cover common cases, but users often have their own domain specific problems. Custom hooks can solve these sorts of problems in a concise and reusable way.

For instance if our domain had a lot of shorthand abbriviations that are not commonly known, we could write a hook to make them long form.
> A similar method to this could be used as a profanity filter

In [ ]:
class AbbreviationHook(HookBase):
    def __init__(self, colname: str = "query"):
        super().__init__(colname=colname, hook_type="pre_processing")
        self.colname = colname

    def _substitute_abbreviations(self, text: str) -> str:
        text = text.lower()
        text = re.sub(r"\bml\b", "machine learning", text)
        text = re.sub(r"\bdev\b", "developer", text)
        text = re.sub(r"\bons\b", "office for national statistics", text)
        return text

    def __call__(self, input_data: VectorStoreSearchInput) -> VectorStoreSearchInput:
        processed_input = input_data.copy()
        processed_input[self.colname] = (
            processed_input[self.colname]
            .astype(str)
            .apply(self._substitute_abbreviations)
        )
        return input_data.__class__.validate(processed_input)


custom_abreviations_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    hooks={"search_preprocess": AbbreviationHook()},  # we add any hooks here.
)

custom_input = make_query_input(["dev", "ml engineer", "ons employee"])

basic_results = basic_vectorstore.search(query=custom_input, n_results=1)

custom_abreviations_results = custom_abreviations_vectorstore.search(
    query=custom_input, n_results=1
)

print("Basic results")
display(basic_results)

print("After custom abreviation hook")
display(custom_abreviations_results)

Basic results


,query_id,query_text,doc_label,doc_text,rank,score
0,0,dev,2134,"Software developer: Designs, writes, and tests...",1,0.746304
1,1,ml engineer,2134,"Software developer: Designs, writes, and tests...",1,0.666152
2,2,ons employee,5241,"Electrician: Installs, maintains, and repairs ...",1,0.597760


After custom abreviation hook


,query_id,query_text,doc_label,doc_text,rank,score
0,0,developer,2134,Web developer: Builds and maintains websites a...,1,0.802484
1,1,machine learning engineer,2134,"Software developer: Designs, writes, and tests...",1,0.702236
2,2,office for national statistics employee,2433,Statistical Officer In Goverment: Analyses dat...,1,0.817491


---
### 6. Add a thresholding hook

Sometimes the best match is still not a good enough match. A thresholding hook lets us set the minimum score we are willing to accept, so low-confidence suggestions could be ignored.

The hook runs after the search and removes results below the threshold. The rest of the search stays exactly the same.

In [ ]:
class ThresholdingHook(HookBase):
    def __init__(self, threshold: float = 0.8):
        super().__init__(colname="score", hook_type="post_processing")
        self.threshold = threshold

    def __call__(self, input_data):
        thresholded_input = input_data[input_data["score"] >= self.threshold]
        return input_data.__class__.validate(thresholded_input)


thresholded_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    hooks={
        "search_postprocess": ThresholdingHook(threshold=0.8)
    },  # we add any hooks here.
)

basic_results = basic_vectorstore.search(query=review_query, n_results=5)
custom_threshold_results = thresholded_vectorstore.search(
    query=review_query,
    n_results=5,
)

print("Basic results")
display(basic_results)

print("After custom threshold hook")
display(custom_threshold_results)

Basic results


,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Vegetable farmer: Cultivates and harvests vege...,1,0.816154
1,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Fruit farmer: Grows and harvests fruits such a...,2,0.801251
2,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Dairy farmer: Manages cows for milk production...,3,0.791010
3,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,"Sheep farmer: Raises sheep for wool, meat, and...",4,0.755043
4,0,Tomato Farmer: Cultivates and harvests tomatoe...,5313,"Carpenter: Constructs, installs, and repairs w...",5,0.661157


After custom threshold hook


,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Vegetable farmer: Cultivates and harvests vege...,1,0.816154
1,0,Tomato Farmer: Cultivates and harvests tomatoe...,5111,Fruit farmer: Grows and harvests fruits such a...,2,0.801251


---

## Take It Further

This demo showed how ClassifAI can turn unstructured occupation responses into ranked, reviewable SOC code suggestions. Along the way, we used metadata, preprocessing hooks, postprocessing hooks, and custom domain logic to make the workflow more useful in practice.

The next step is to get it running on your own machine with your own knowledgebase. Our Github is the best place to start that journey.

<div style="margin: 1.5em 0; padding: 1.2em 1.4em; border-left: 5px solid #003078; background-color: #f2f2f2;">
    <strong>Explore the ClassifAI project</strong><br>
    Read the documentation, source code, and examples on GitHub.
    <br><br>
    <a href="https://github.com/datasciencecampus/classifai" style="display: inline-block; padding: 0.65em 1em; color: white; background-color: #003078; text-decoration: none; border-radius: 4px; font-weight: 600;">View ClassifAI on GitHub &#8594;</a>
</div>